# CODE-M · Day 3 · Hands-on A
## Beyond one FBA solution: FVA, pFBA and robustness

> **Student version.** Fill in the ✍️ cells. A complete version with all solutions is available from the instructors.

On Day 1 FBA gave you one growth rate and one flux distribution. The growth rate is a firm prediction; the flux distribution often is not, because many distributions can give the same growth. This notebook asks four questions that a single FBA run can't answer:

1. **Is the solution unique?** Alternative optima and loops (§1).
2. **How certain is each flux?** Flux variability analysis, FVA (§2).
3. **Which solution should I report?** Parsimonious FBA, pFBA (§3).
4. **How does the optimum respond to the environment?** Robustness analysis and phase planes, and what changes without oxygen (§4–6).

You'll use two curated models: **E. coli core** (95 reactions, the model from Day 1, small enough to see every reaction) and **iMM904**, the genome-scale model of the yeast *Saccharomyces cerevisiae*.

| Section | Model | Time |
|---|---|---|
| 0 · Setup | | 3 min |
| 1 · One growth rate, many flux distributions | E. coli core | 10 min |
| 2 · FVA: how certain is each flux? | E. coli core | 12 min |
| 3 · pFBA: the most economical solution | iMM904 | 10 min |
| 4 · Robustness: what oxygen does | E. coli core | 12 min |
| 5 · Phase plane: glucose and oxygen together | E. coli core | 8 min |
| 6 · Yeast with and without oxygen | iMM904 | 15 min |

**How to use this notebook**
- Run the cells in order with **Shift + Enter**.
- ✍️ marks a task for you: replace each `...` with your code.
- ✅ marks a checkpoint: if it prints a green tick, carry on. If it raises an error, the message tells you what to look at.
- 💬 marks a question to discuss with your neighbour.
- Short of time? §5 is optional.

## 0 · Setup  ·  3 min
Run this cell to load the tools. `day3_helpers.py`, in the same folder as this notebook, knows where the model files are and draws optional Escher maps; you don't need to open it.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger("cobra").setLevel(logging.ERROR)      # keep the output tidy

import pandas as pd
import matplotlib.pyplot as plt
import cobra
from cobra.flux_analysis import pfba, flux_variability_analysis, production_envelope
from day3_helpers import MODELS, flux_map

cobra.Configuration().processes = 1                      # small models: no need for parallel workers
print("cobrapy", cobra.__version__, "| LP solver:", cobra.Configuration().solver.__name__.split("_")[0].split(".")[-1])

## 1 · One growth rate, many flux distributions  ·  10 min

Start with E. coli core on its default medium: glucose (at most 10 mmol/gDW/h) and plenty of oxygen.

In [ ]:
ecoli = cobra.io.read_sbml_model(str(MODELS / "e_coli_core.xml"))
BIO = "BIOMASS_Ecoli_core_w_GAM"
ecoli

✍️ Run FBA and store the solution in `sol`. The print line then shows the growth rate and the oxygen uptake (exchange reaction `EX_o2_e`; uptake fluxes are negative).

In [ ]:
sol = ...                                    # ✍️ run FBA on ecoli
print(f"growth: {sol.objective_value:.3f} per hour | oxygen uptake: {-sol.fluxes['EX_o2_e']:.1f} mmol/gDW/h")

In [ ]:
# ✅ Checkpoint
assert abs(sol.objective_value - 0.874) < 0.01, "growth should be about 0.874 per hour"
print("✅ FBA done")

Is this flux distribution the only one with that growth rate? **Flux variability analysis (FVA)** answers the question: for every reaction it finds the minimum and maximum flux that still allow (a fraction of) the optimal growth. With `fraction_of_optimum=1.0`, growth must stay exactly optimal. Here are the reactions whose flux can change:

In [ ]:
fva100 = flux_variability_analysis(ecoli, fraction_of_optimum=1.0)
fva100[(fva100.maximum - fva100.minimum) > 1e-6]

Only two reactions can change, but one of them can carry up to 1,000 units of flux at no cost to growth. Read them:

In [ ]:
for rid in ["SUCDi", "FRD7"]:
    r = ecoli.reactions.get_by_id(rid)
    print(f"{rid:6s} {r.name}:\n       {r.build_reaction_string(use_metabolite_names=True)}")

Succinate dehydrogenase (`SUCDi`) and fumarate reductase (`FRD7`) can run at the same time, turning succinate into fumarate and back while the quinones cycle between their oxidised and reduced forms. The net effect is zero, so FBA has no reason to prefer 0 or 1,000. Such a **loop** is thermodynamically impossible in a real cell. *Loopless* FVA forbids loops:

In [ ]:
fva_loopless = flux_variability_analysis(ecoli, fraction_of_optimum=1.0, loopless=True)
print("reactions that can vary at maximum growth, loops removed:", ((fva_loopless.maximum - fva_loopless.minimum) > 1e-6).sum())

💬 In E. coli core the optimal flux distribution is unique once the loop is removed. Do you expect the same in a genome-scale model with 1,500 reactions? (You'll check in §3.)

## 2 · FVA: how certain is each flux?  ·  12 min

Cells rarely grow at exactly their theoretical maximum. Allowing growth to drop to 90% of the optimum shows how much freedom the network really has.

✍️ Run FVA with `fraction_of_optimum=0.9` and store it in `fva90`. Then keep, in `variable`, the reactions whose range is wider than `1e-6`, and in `can_be_zero` those whose range includes zero (minimum ≤ 0 ≤ maximum).

In [ ]:
fva90 = ...                                                       # ✍️ FVA at 90% of the optimum
variable = fva90[...]                                             # ✍️ range (maximum - minimum) wider than 1e-6
can_be_zero = fva90[(fva90.minimum <= 1e-9) & (fva90.maximum >= -1e-9)]
print(f"{len(variable)} of {len(ecoli.reactions)} reactions can vary; {len(can_be_zero)} can be switched off")

In [ ]:
# ✅ Checkpoint
assert 50 < len(variable) < len(ecoli.reactions), "check fva90 and the condition for `variable`"
print("✅ FVA at 90% done")

At the optimum E. coli core secretes nothing but CO₂ and water. What could it secrete at 90% of the optimum?

In [ ]:
products = {"EX_ac_e": "acetate", "EX_etoh_e": "ethanol", "EX_for_e": "formate", "EX_lac__D_e": "D-lactate", "EX_succ_e": "succinate"}
ranges = fva90.loc[list(products)]
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.barh(list(products.values()), ranges.maximum - ranges.minimum, left=ranges.minimum, color="#1B998B")
ax.set_xlabel("possible secretion at ≥ 90% of maximum growth (mmol/gDW/h)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

And how quickly does the freedom grow as growth drops below the optimum?

In [ ]:
fractions = [1.0, 0.999, 0.99, 0.95, 0.9, 0.8]
n_variable = {f: int(((r.maximum - r.minimum) > 1e-6).sum())
              for f, r in ((f, flux_variability_analysis(ecoli, fraction_of_optimum=f)) for f in fractions)}
pd.Series(n_variable, name="reactions that can vary").rename_axis("fraction of optimal growth")

💬 The FBA solution in §1 has zero flux through pyruvate formate-lyase (`PFL`). Look at its range in `fva90`. Would you report "PFL is off" as a prediction of the model?

In [ ]:
fva90.loc[["PFL", "PDH", "CS", "EX_o2_e"]].round(2)

## 3 · pFBA: the most economical solution  ·  10 min

Now a genome-scale model: **iMM904**, the yeast *S. cerevisiae*. Its default medium caps oxygen uptake at 2 mmol/gDW/h; keep that in mind for §6.

In [ ]:
yeast = cobra.io.read_sbml_model(str(MODELS / "iMM904.xml"))
YBIO = "BIOMASS_SC5_notrace"
print(f"{yeast.id}: {len(yeast.reactions)} reactions, {len(yeast.metabolites)} metabolites, {len(yeast.genes)} genes")
pd.Series(yeast.medium, name="maximum uptake")

**Parsimonious FBA (pFBA)** first maximises growth, then, among all flux distributions with that growth, picks the one with the smallest total flux: the sum of the absolute values of all fluxes. The assumption is that a cell avoids making more enzyme than it needs.

✍️ With unlimited oxygen, compare FBA and pFBA. Complete `flux_summary()` so that it also returns the total absolute flux and the number of active reactions (absolute flux above `1e-6`). A pandas Series has `.abs()` and `.sum()`.

In [ ]:
def flux_summary(fluxes):
    return {"growth": round(fluxes[YBIO], 3),
            "total |flux|": ...,                         # ✍️ sum of the absolute fluxes, rounded to 1 decimal
            "active reactions": ...}                     # ✍️ number of fluxes with absolute value above 1e-6


with yeast:
    yeast.reactions.EX_o2_e.lower_bound = -1000         # unlimited oxygen, only inside this block
    fba_fluxes = yeast.optimize().fluxes
    pfba_fluxes = pfba(yeast).fluxes

pd.DataFrame({"FBA": flux_summary(fba_fluxes), "pFBA": flux_summary(pfba_fluxes)})

In [ ]:
# ✅ Checkpoint
s_fba, s_pfba = flux_summary(fba_fluxes), flux_summary(pfba_fluxes)
assert abs(s_fba["growth"] - s_pfba["growth"]) < 1e-3, "pFBA should keep the same growth"
assert s_pfba["total |flux|"] <= s_fba["total |flux|"] + 1e-6, "pFBA should never use more total flux than FBA"
print("✅ Same growth, less total flux")

Now the question from §1: how many reactions can vary at maximum growth in a genome-scale model? (This FVA takes about 15 seconds.)

In [ ]:
with yeast:
    yeast.reactions.EX_o2_e.lower_bound = -1000
    fva_yeast = flux_variability_analysis(yeast, fraction_of_optimum=1.0)
span = fva_yeast.maximum - fva_yeast.minimum
print(f"{(span > 1e-6).sum()} reactions can vary at maximum growth; {(span > 1000).sum()} of them practically without limit")
fva_yeast[span > 1000].head(8)

## 4 · Robustness: what oxygen does  ·  12 min

**Robustness analysis** changes one constraint step by step and records the optimum. Back to E. coli core: how do growth and by-products respond to the oxygen supply?

✍️ Complete the loop: for each oxygen uptake limit, set the lower bound of `EX_o2_e` to `-o2`, run pFBA, and store growth and by-products. (pFBA gives unique, loop-free by-product fluxes.)

In [ ]:
rows = []
for o2 in range(0, 31, 2):
    with ecoli:
        ...                                            # ✍️ set the lower bound of EX_o2_e to -o2
        f = ...                                        # ✍️ run pFBA and keep its fluxes
        rows.append({"oxygen": o2, "growth": f[BIO], **{name: f[ex] for ex, name in products.items()}})
o2_scan = pd.DataFrame(rows).set_index("oxygen").round(3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
o2_scan["growth"].plot(ax=ax1, marker="o", color="#0E3B43")
ax1.set_ylabel("growth (per hour)")
o2_scan[list(products.values())].plot(ax=ax2, marker=".")
ax2.set_ylabel("secretion (mmol/gDW/h)")
for ax in (ax1, ax2):
    ax.set_xlabel("maximum oxygen uptake (mmol/gDW/h)")
plt.tight_layout()
plt.show()

In [ ]:
# ✅ Checkpoint
assert abs(o2_scan.loc[0, "growth"] - 0.212) < 0.01 and abs(o2_scan.loc[30, "growth"] - 0.874) < 0.01, "check the bound and the pFBA call"
print("✅ Oxygen scan done")

💬 Describe the three regimes you see. Which by-products does E. coli make without oxygen? At which oxygen supply does acetate peak? From which oxygen supply does more oxygen stop helping?

**See it on a map (optional).** The cell below writes two Escher maps of E. coli core fluxes, with and without oxygen, as HTML files you can open in a browser. If Escher isn't installed it just says so. The table shows the same comparison in numbers.

In [ ]:
aerobic = pfba(ecoli).fluxes
with ecoli:
    ecoli.reactions.EX_o2_e.lower_bound = 0
    anaerobic = pfba(ecoli).fluxes
flux_map(aerobic, "ecoli_core_aerobic.html")
flux_map(anaerobic, "ecoli_core_anaerobic.html")
key = ["EX_glc__D_e", "EX_o2_e", "PDH", "PFL", "CS", "AKGDH", "CYTBD", "ATPS4r", "EX_ac_e", "EX_etoh_e", "EX_for_e", BIO]
pd.DataFrame({"aerobic": aerobic, "anaerobic": anaerobic}).loc[key].round(2)

## 5 · Phase plane: glucose and oxygen together  ·  8 min  (optional)

A **phenotypic phase plane** varies two constraints at once. `production_envelope()` computes the maximum growth on a grid of glucose and oxygen uptakes.

In [ ]:
ppp = production_envelope(ecoli, ["EX_glc__D_e", "EX_o2_e"], objective=BIO, points=20)
grid = ppp.pivot_table(index="EX_o2_e", columns="EX_glc__D_e", values="flux_maximum")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
mesh = ax.pcolormesh(-grid.columns, -grid.index, grid.values, cmap="viridis", shading="auto")
fig.colorbar(mesh, label="maximum growth (per hour)")
ax.set_xlabel("glucose uptake (mmol/gDW/h)")
ax.set_ylabel("oxygen uptake (mmol/gDW/h)")
plt.tight_layout()
plt.show()

Each square forces exactly that glucose and oxygen uptake. For each glucose uptake there is an oxygen uptake above which more oxygen no longer helps; forcing in even more oxygen wastes carbon on burning it, so growth falls again. Joining these points gives the **line of optimality**: the oxygen-to-glucose ratio at which each glucose molecule yields the most biomass.

✍️ From `o2_scan` (glucose 10 mmol/gDW/h), find the smallest oxygen supply at which growth reaches 99.9% of its maximum.

In [ ]:
o2_needed = ...                                      # ✍️ first index of o2_scan where growth >= 0.999 * its maximum
print(f"growth saturates at an oxygen supply of {o2_needed} mmol/gDW/h, an O2:glucose ratio of about {o2_needed / 10:.1f}")

In [ ]:
# ✅ Checkpoint
assert 20 <= o2_needed <= 24, "growth should saturate at about 22 mmol O2/gDW/h"
print("✅ You found a point on the line of optimality")

## 6 · Yeast with and without oxygen  ·  15 min

### 6.1 · The oxygen cap
iMM904's default medium caps oxygen uptake at 2 mmol/gDW/h. Why?

✍️ Complete `yeast_run()`: inside the `with` block, set the lower bound of `EX_o2_e` to `-o2` and run pFBA. The table compares the default cap with unlimited oxygen.

In [ ]:
def yeast_run(o2):
    with yeast:
        ...                                          # ✍️ set the lower bound of EX_o2_e to -o2
        f = ...                                      # ✍️ run pFBA and keep its fluxes
        return {"growth": round(f[YBIO], 3), "ethanol": round(f["EX_etoh_e"], 2), "oxygen used": round(-f["EX_o2_e"], 2)}


pd.DataFrame({"oxygen ≤ 2 (default)": yeast_run(2), "oxygen unlimited": yeast_run(1000)})

In [ ]:
# ✅ Checkpoint
assert yeast_run(1000)["ethanol"] < 0.01 < yeast_run(2)["ethanol"], "with unlimited oxygen the model should make no ethanol"
print("✅ Oxygen cap explored")

💬 In a flask with plenty of glucose and air, *S. cerevisiae* makes a lot of ethanol (the **Crabtree effect**). Which column matches that? What is the oxygen cap in the default medium really doing?

### 6.2 · No oxygen at all

In [ ]:
with yeast:
    yeast.reactions.EX_o2_e.lower_bound = 0
    print(f"anaerobic growth: {yeast.slim_optimize(error_value=0):.3f} per hour")

Yeast grows anaerobically in the lab, but the model doesn't. As on Day 2, find out why: test each biomass component with a temporary demand reaction, and list those that can't be made.

✍️ Inside the inner `with` block, make the demand reaction the objective, and add the metabolite to `missing` if the maximum demand flux is (practically) zero.

In [ ]:
missing = []
with yeast:
    yeast.reactions.EX_o2_e.lower_bound = 0
    for met, coefficient in yeast.reactions.get_by_id(YBIO).metabolites.items():
        if coefficient < 0:                                   # a biomass component
            with yeast:
                demand = yeast.add_boundary(met, type="demand")
                ...                                           # ✍️ make `demand` the objective
                if ...:                                       # ✍️ maximum demand flux below 1e-9 (use slim_optimize(error_value=0))
                    missing.append((met.id, met.name))
pd.DataFrame(missing, columns=["id", "name"])

In [ ]:
# ✅ Checkpoint
assert "ergst_c" in [m for m, _ in missing], "ergosterol should be among the components that can't be made without oxygen"
print(f"✅ {len(missing)} biomass components can't be made without oxygen")

Sterols (ergosterol, zymosterol) and the yeast membrane lipids can't be made. Give the model what yeast physiologists give real anaerobic cultures: sterols and fatty acids in the medium.

In [ ]:
SUPPLEMENTS = [r.id for r in yeast.exchanges
               if any(k in r.id for k in ("ergst", "zymst", "lanost", "hdca", "hdcea", "ocdca", "ocdcea", "ocdcya", "ttdca"))]
with yeast:
    yeast.reactions.EX_o2_e.lower_bound = 0
    for rid in SUPPLEMENTS:
        yeast.reactions.get_by_id(rid).lower_bound = -1000
    f = pfba(yeast).fluxes
print(f"anaerobic, with sterols and fatty acids: growth {f[YBIO]:.3f} per hour, ethanol {f['EX_etoh_e']:.1f} mmol/gDW/h")
print("supplements taken up:", [rid for rid in SUPPLEMENTS if f[rid] < -1e-9])

💬 Why can't yeast make sterols and unsaturated fatty acids without oxygen? What does this tell you about using a model to design an anaerobic fermentation?

## Recap

| Question | Method | What you found |
|---|---|---|
| Is the FBA solution unique? | FVA at 100% of the optimum | In E. coli core almost, apart from a loop; in iMM904 about 180 reactions are free |
| How certain is each flux? | FVA at 90% | Most fluxes have wide ranges; report ranges, not single values |
| Which solution to report? | pFBA | The cheapest flux distribution: same growth, no loops |
| How does growth respond to the environment? | Robustness analysis, phase plane | Three regimes in oxygen; a line of optimality at about 2.2 O₂ per glucose |
| What does the model miss? | Comparing with physiology | No Crabtree effect without an oxygen cap; anaerobic yeast needs sterols and fatty acids |

**At 11:00** you'll use the same tools to ask which genes a cell can't live without, and how to rewire it to make a product.